# E1--E4 release environment check

This notebook performs the CPU-safe checks that should be run immediately
after unpacking the collaborator archive. It validates package versions,
the four experiment/config/notebook contracts, frozen result completeness,
and a focused CPU regression suite. It does **not** launch a production GPU
experiment.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if not (ROOT / "src" / "manuscript.py").is_file():
    raise RuntimeError("Run this notebook from JCP_experiments/notebooks")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))
sys.path.insert(0, str(ROOT))

import matplotlib
import nbclient
import nbformat
import numpy
import pandas
import scipy
import torch
import yaml

print("Project root:", ROOT)
print("Python:", sys.version.split()[0], platform.platform())
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

In [ ]:
from scripts.validate_release import validate_release

report = validate_release(
    ROOT,
    check_results=True,
    require_figures=True,
)
print("Release validation:", report["status"])
for key, item in report["experiments"].items():
    print(key, item["results"]["release_methods"])

In [ ]:
command = [
    sys.executable, "-m", "pytest", "-q",
    "tests_cpu/test_release_structure.py",
    "tests_cpu/test_manuscript_replot.py",
    "tests_cpu/test_runner_controls.py",
    "tests_cpu/test_sampler_diagnostics.py",
    "tests_cpu/test_stationarity.py",
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=ROOT, check=True)
print("CPU release checks passed.")